# 🔬 CellOneFormer: Entraînement sur le VRAI Jeu de Données Multi-Modalité
### Benchmark basé sur *The Multi-modality Cell Segmentation Challenge (Nature Methods 2024)*

Ce notebook permet de :
1. **Synchroniser les vraies données de microscopie** (`DatasetCellSeg/Training` : 1000 images & masques annotés) vers le SSD local de Colab.
2. **Découper les grandes images en tuiles $512 \times 512$** avec filtrage des tuiles vides pour maximiser l'apprentissage sur les vraies cellules.
3. **Entraîner CellOneFormer (Swin-Transformer)** avec précision mixte `fp16` et accumulation de gradients (conçu spécialement pour le GPU T4 15 Go sans saturation VRAM).
4. **Sauvegarder les poids entraînés** directement sur votre Google Drive dans `MyDrive/cell_oneformer_real_checkpoints/`.
5. **Évaluer et calibrer les seuils** sur le jeu de validation (`Tuning/` : 101 images) avec les métriques officielles : $\text{AP}_{50}$, $\text{AP}_{75}$, $\text{mAP}_{[50:95]}$, $\text{F1}$.
6. **Visualiser la segmentation d'instances reconstituée** sur les images de test réelles.

## 1. Vérification du GPU & Montage de Google Drive

In [ ]:
!nvidia-smi

from google.colab import drive
import os

drive.mount('/content/drive')
print("✅ Google Drive connecté avec succès.")

## 2. Configuration du Code Source & Dépendances

In [ ]:
import os
import sys

os.makedirs("/content/CellOneFormer", exist_ok=True)

# Synchronisation du code source depuis Drive ou GitHub
if os.path.exists("/content/drive/MyDrive/projet/src"):
    print("📁 Synchronisation des scripts depuis Google Drive (MyDrive/projet)...\n")
    !cp -ru /content/drive/MyDrive/projet/* /content/CellOneFormer/
elif not os.path.exists("/content/CellOneFormer/src"):
    print("📥 Clonage du dépôt CellOneFormer depuis GitHub...\n")
    !git clone https://github.com/BaptFontaine/CellOneFormer.git /tmp/repo
    !cp -ru /tmp/repo/* /content/CellOneFormer/

%cd /content/CellOneFormer
%env PYTHONPATH=/content/CellOneFormer

if "/content/CellOneFormer" not in sys.path:
    sys.path.insert(0, "/content/CellOneFormer")

# Optimisation de la mémoire GPU
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Installation propre en conservant le NumPy 2.x natif de Colab (Python 3.13)
!pip install -q "numpy>=2.0.0,<2.3.0" transformers accelerate albumentations tifffile imagecodecs
print("✅ Environnement CellOneFormer prêt sans conflit NumPy !")

## 3. Synchronisation du Vrai Dataset (`Training/` et `Tuning/`)
Copie les images de microscopie réelles de Google Drive vers le SSD rapide de Google Colab.

In [ ]:
import os
import glob

DRIVE_DATASET = "/content/drive/MyDrive/DatasetCellSeg"
LOCAL_RAW = "/content/CellOneFormer/data/raw/DatasetCellSeg"
os.makedirs(LOCAL_RAW, exist_ok=True)

print(f"📁 Synchronisation depuis {DRIVE_DATASET}...")
# Copie avec respect de la casse (Training, Tuning, Testing)
for folder in ["Training", "Tuning", "Testing"]:
    src = os.path.join(DRIVE_DATASET, folder)
    dst = os.path.join(LOCAL_RAW, folder)
    if os.path.exists(src):
        os.makedirs(dst, exist_ok=True)
        print(f"   Copie de {folder}/...")
        !cp -ru "{src}"/* "{dst}/"

# Inventaire des données prêtes sur le SSD local
train_imgs = sorted(glob.glob(f"{LOCAL_RAW}/Training/images/*.*") + glob.glob(f"{LOCAL_RAW}/training/images/*.*"))
train_masks = sorted(glob.glob(f"{LOCAL_RAW}/Training/labels/*.*") + glob.glob(f"{LOCAL_RAW}/training/labels/*.*"))
tuning_imgs = sorted(glob.glob(f"{LOCAL_RAW}/Tuning/images/*.*") + glob.glob(f"{LOCAL_RAW}/tuning/images/*.*"))
tuning_masks = sorted(glob.glob(f"{LOCAL_RAW}/Tuning/labels/*.*") + glob.glob(f"{LOCAL_RAW}/tuning/labels/*.*"))
test_imgs = sorted(glob.glob(f"{LOCAL_RAW}/Testing/**/*.*", recursive=True))
test_imgs = [f for f in test_imgs if f.lower().endswith(('.png', '.tif', '.tiff', '.jpg', '.jpeg'))]

print(f"\n📊 Données synchronisées avec succès sur le SSD local :")
print(f"   - Training : {len(train_imgs)} images & {len(train_masks)} masques d'instances réels")
print(f"   - Tuning   : {len(tuning_imgs)} images & {len(tuning_masks)} masques de validation")
print(f"   - Testing  : {len(test_imgs)} images de test final")

## 4. Découpage en Tuiles Glissantes ($512 \times 512$)
Découpe les images de microscopie haute résolution en tuiles de $512 \times 512$ (recouvrement 64 px).

> **Option d'échantillonnage rapide :**
> Pour un entraînement ultra-rapide (~10-15 minutes), on peut tuiler un sous-ensemble de 100 images (produisant ~400-500 tuiles), ou tuiler les 1000 images pour un entraînement complet.

In [ ]:
import yaml
import shutil
from pathlib import Path

%cd /content/CellOneFormer
%env PYTHONPATH=/content/CellOneFormer

# 1. Lier les dossiers Training et Tuning pour que le script trouve les vraies donnees sans conflit de casse
synth_train = Path("data/raw/DatasetCellSeg/training")
real_train = Path("data/raw/DatasetCellSeg/Training")
if synth_train.exists() and not synth_train.is_symlink() and synth_train != real_train:
    shutil.rmtree(synth_train)

!ln -sfn /content/CellOneFormer/data/raw/DatasetCellSeg/Training /content/CellOneFormer/data/raw/DatasetCellSeg/training
!ln -sfn /content/CellOneFormer/data/raw/DatasetCellSeg/Tuning /content/CellOneFormer/data/raw/DatasetCellSeg/tuning
!ln -sfn /content/CellOneFormer/data/raw/DatasetCellSeg/Testing /content/CellOneFormer/data/raw/DatasetCellSeg/testing

processed_dir = Path("data/processed")
if processed_dir.exists():
    shutil.rmtree(processed_dir)
processed_dir.mkdir(parents=True, exist_ok=True)

# 2. Charger la configuration et définir le nombre d'images à tuiler
with open("configs/config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

# Nombre d'images d'entraînement à utiliser (ex: 100 pour 15 min, ou None pour les 1000)
MAX_TRAIN_IMAGES = 150  # Modifiable selon le temps souhaité (150 images = ~600 tuiles)
cfg["data"]["max_train_samples"] = MAX_TRAIN_IMAGES
cfg["data"]["max_val_samples"] = 30

# 3. Lancement du tuilage sur les vraies images
from src.training.train import prepare_tiled_data
print(f"🔪 Découpage en tuiles des {MAX_TRAIN_IMAGES} premières images de Training/...")
prepare_tiled_data(cfg)

n_train_tiles = len(glob.glob("data/processed/train/images/*.png"))
n_val_tiles = len(glob.glob("data/processed/val/images/*.png"))
print(f"\n✅ Prétraitement terminé : {n_train_tiles} tuiles d'entraînement et {n_val_tiles} tuiles de validation générées !")

## 5. Lancement de l'Entraînement OneFormer (5 Époques puis 20 Époques)
- Architecture : Swin-Tiny + OneFormer Universal Segmentation (`150` queries)
- `batch_size: 2`, `gradient_accumulation_steps: 8` (Lot effectif = 16, ~7 Go VRAM sur GPU T4)
- Validation automatique `F1@0.5` à chaque époque et sauvegarde continue sur Google Drive.
- Si un checkpoint existe déjà sur Google Drive, `--resume` reprend automatiquement l'entraînement sans repartir de zéro.

In [ ]:
%cd /content/CellOneFormer
%env PYTHONPATH=/content/CellOneFormer

import os

DRIVE_CKPT = "/content/drive/MyDrive/cell_oneformer_real_checkpoints/oneformer_real_cells_best.pt"

# Choisissez 5 pour le premier palier rapide, puis 20 pour l'entraînement complet :
EPOCHS = 5  # Passez à 20 pour le second run complet

resume_flag = f"--resume {DRIVE_CKPT}" if os.path.exists(DRIVE_CKPT) else ""
print(f"🚀 Démarrage de l'entraînement pour {EPOCHS} époques sur GPU ({('Reprise depuis checkpoint' if resume_flag else 'Nouveau run')})...")

!python3 src/training/train.py \
    --config configs/config.yaml \
    --epochs {EPOCHS} \
    --batch-size 2 \
    --gradient-accumulation-steps 8 \
    --mixed-precision fp16 \
    --device cuda {resume_flag}


## 6. Sauvegarde Définitive des Poids Réels sur Google Drive

In [ ]:
import os
import shutil

DRIVE_SAVE_DIR = "/content/drive/MyDrive/cell_oneformer_real_checkpoints"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

best_ckpt = "checkpoints/oneformer_cells_best.pt"
if os.path.exists(best_ckpt):
    dest_ckpt = os.path.join(DRIVE_SAVE_DIR, "oneformer_real_cells_best.pt")
    shutil.copy(best_ckpt, dest_ckpt)
    size_mb = os.path.getsize(dest_ckpt) / (1024 * 1024)
    print(f"✅ Modèle sauvegardé avec succès sur Google Drive :\n   👉 {dest_ckpt} ({size_mb:.1f} Mo)")
else:
    print("⚠️ Checkpoint local non trouvé dans checkpoints/")

## 7. Évaluation & Tuning des Seuils sur le Jeu `Tuning/` (101 Images Réelles)
Calcule les métriques officielles du challenge : **AP@50**, **AP@75**, **mAP@[50:95]** et **F1**.

In [ ]:
%cd /content/CellOneFormer
%env PYTHONPATH=/content/CellOneFormer

import os
import glob
import cv2
import yaml
import torch
import numpy as np
from PIL import Image

LOCAL_RAW = "/content/CellOneFormer/data/raw/DatasetCellSeg"
tuning_imgs = sorted(glob.glob(f"{LOCAL_RAW}/Tuning/images/*.*") + glob.glob(f"{LOCAL_RAW}/tuning/images/*.*"))
tuning_masks = sorted(glob.glob(f"{LOCAL_RAW}/Tuning/labels/*.*") + glob.glob(f"{LOCAL_RAW}/tuning/labels/*.*"))
test_imgs = sorted([f for f in glob.glob(f"{LOCAL_RAW}/Testing/**/*.*", recursive=True) if f.lower().endswith(('.png', '.tif', '.tiff', '.jpg', '.jpeg'))])

from src.dataset.tiler import SlidingWindowTiler, load_microscopy_image, load_instance_mask
from src.inference.stitcher import TileStitcher
from src.models.oneformer_wrapper import CellOneFormerModule
from src.utils.metrics import compute_challenge_benchmark_metrics

CHECKPOINT = "/content/drive/MyDrive/cell_oneformer_real_checkpoints/oneformer_real_cells_best.pt"
if not os.path.exists(CHECKPOINT):
    CHECKPOINT = "checkpoints/oneformer_cells_best.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
with open("configs/config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

# Initialisation du modèle en mode inférence
model = CellOneFormerModule(
    pretrained_model_name=cfg["model"]["pretrained_model_name"],
    num_classes=cfg["model"]["num_classes"],
    num_queries=cfg["model"]["num_queries"],
    is_training=False,
)
model.model.is_training = False
model.model.model.is_training = False
model.model.model.text_mapper = None

if os.path.exists(CHECKPOINT):
    model.load_state_dict(torch.load(CHECKPOINT, map_location=device), strict=False)
    print(f"✅ Poids réels chargés : {CHECKPOINT}")

model.to(device)
model.eval()

# Évaluation sur 5 images de validation complètes
tiler = SlidingWindowTiler(tile_size=512, tile_overlap=64)
tuning_results = []
stitcher = TileStitcher(iou_threshold=0.5, min_area=15, max_area=8000)

print("\n🔍 Évaluation sur les images réelles de validation...")
for score_th in [0.2, 0.3, 0.4]:
    sample_metrics = []
    for img_p in tuning_imgs[:5]:
        stem = os.path.splitext(os.path.basename(img_p))[0]
        mask_matches = [m for m in tuning_masks if stem in os.path.basename(m)]
        if not mask_matches:
            continue

        image = load_microscopy_image(img_p)
        gt_mask = load_instance_mask(mask_matches[0])
        gt_instances = [gt_mask == gid for gid in np.unique(gt_mask) if gid > 0]
        if not gt_instances:
            continue
        gt_instances = np.array(gt_instances)

        h, w = image.shape[:2]
        tiles = tiler.tile_image_and_mask(image)
        tile_results = []

        with torch.no_grad():
            for tile in tiles:
                pil_img = Image.fromarray(tile["image"])
                inputs = model.processor(images=pil_img, task_inputs=["instance"], return_tensors="pt")
                dev_in = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}
                outputs = model(**dev_in)

                tile_h, tile_w = tile["image"].shape[:2]
                post = model.post_process_instances(outputs, target_sizes=[(tile_h, tile_w)], threshold=score_th)

                t_masks = post[0].get("masks", []) if post else []
                t_scores = post[0].get("scores", [1.0] * len(t_masks)) if post else []
                tile_results.append({"masks": t_masks, "scores": t_scores, "coords": tile["coords"]})

        _, kept_masks = stitcher.stitch_tile_predictions(h, w, tile_results)
        m = compute_challenge_benchmark_metrics(kept_masks, gt_instances)
        sample_metrics.append(m)

    if sample_metrics:
        tuning_results.append({
            "score_threshold": score_th,
            "AP@50": float(np.mean([x.get("AP@50", x.get("AP_50", 0.0)) for x in sample_metrics])),
            "AP@75": float(np.mean([x.get("AP@75", x.get("AP_75", 0.0)) for x in sample_metrics])),
            "mAP@[50:95]": float(np.mean([x.get("mAP@[50:95]", x.get("mAP_50_95", 0.0)) for x in sample_metrics])),
            "mean_f1": float(np.mean([x.get("mean_f1", x.get("F1_50", 0.0)) for x in sample_metrics])),
        })

if tuning_results:
    tuning_results.sort(key=lambda x: x.get("mAP@[50:95]", 0.0), reverse=True)
    print("\n" + "=" * 65)
    print(f" {'Score Th':<10} | {'AP@50':<12} | {'AP@75':<12} | {'mAP@[50:95]':<14} | {'F1':<8}")
    print("-" * 65)
    for r in tuning_results:
        print(f" {r['score_threshold']:<10.2f} | {r['AP@50']:<12.4f} | {r['AP@75']:<12.4f} | {r['mAP@[50:95]']:<14.4f} | {r['mean_f1']:<8.4f}")
    print("=" * 65)
    best_thresh = tuning_results[0]["score_threshold"]
else:
    best_thresh = 0.5

print(f"\n⭐ Meilleur seuil de détection sélectionné : {best_thresh}")

## 8. Inférence Pleine Résolution & Rendu Visuel Haute Qualité
Affiche côte-à-côte l'image brute, les masques d'instances et la superposition avec contours nets.

In [ ]:
import os
import glob
import cv2
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from src.dataset.tiler import SlidingWindowTiler, load_microscopy_image
from src.inference.stitcher import TileStitcher
from src.models.oneformer_wrapper import CellOneFormerModule

LOCAL_RAW = "/content/CellOneFormer/data/raw/DatasetCellSeg"
tuning_imgs = sorted(glob.glob(f"{LOCAL_RAW}/Tuning/images/*.*"))
test_imgs = sorted([f for f in glob.glob(f"{LOCAL_RAW}/Testing/**/*.*", recursive=True) if f.lower().endswith(('.png', '.tif', '.tiff', '.jpg', '.jpeg'))])

OUTPUT_PRED_DIR = "/content/CellOneFormer/predictions_real"
os.makedirs(OUTPUT_PRED_DIR, exist_ok=True)

CHECKPOINT = "/content/drive/MyDrive/cell_oneformer_real_checkpoints/oneformer_real_cells_best.pt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
with open("configs/config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

if "model" not in globals():
    model = CellOneFormerModule(
        pretrained_model_name=cfg["model"]["pretrained_model_name"],
        num_classes=cfg["model"]["num_classes"],
        num_queries=cfg["model"]["num_queries"],
        is_training=False,
    )
    model.model.is_training = False
    model.model.model.is_training = False
    if os.path.exists(CHECKPOINT):
        model.load_state_dict(torch.load(CHECKPOINT, map_location=device), strict=False)
    model.to(device).eval()

tiler = SlidingWindowTiler(tile_size=512, tile_overlap=64)
stitcher = TileStitcher(iou_threshold=0.5, min_area=15)
np.random.seed(42)
palette = np.random.randint(60, 255, size=(5000, 3), dtype=np.uint8)
palette[0] = [0, 0, 0]

targets = (test_imgs + tuning_imgs)[:10]
print(f"🚀 Inférence directe en mémoire GPU sur {len(targets)} images réelles...")

for idx, img_path in enumerate(targets, 1):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    image = load_microscopy_image(img_path)
    if image.ndim == 2:
        image = np.stack([image] * 3, axis=-1)
    elif image.shape[-1] > 3:
        image = image[..., :3]
    h, w = image.shape[:2]

    tiles = tiler.tile_image_and_mask(image)
    tile_results = []
    with torch.no_grad():
        for tile in tiles:
            pil_img = Image.fromarray(tile["image"])
            inputs = model.processor(images=pil_img, task_inputs=["instance"], return_tensors="pt")
            dev_in = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}
            outputs = model(**dev_in)
            th, tw = tile["image"].shape[:2]
            probs = torch.softmax(outputs.class_queries_logits[0], dim=-1)
            cell_scores = torch.maximum(probs[:, 0], probs[:, 1])
            up_masks = torch.nn.functional.interpolate(outputs.masks_queries_logits, size=(th, tw), mode="bilinear", align_corners=False)[0].sigmoid()
            t_masks, t_scores = [], []
            for q_idx in torch.argsort(cell_scores, descending=True):
                s = float(cell_scores[q_idx].item())
                if s < 0.25:
                    break
                m = (up_masks[q_idx] >= 0.5).detach().cpu().numpy()
                if int(m.sum()) >= 15:
                    t_masks.append(m)
                    t_scores.append(s)
            tile_results.append({"masks": t_masks, "scores": t_scores, "coords": tile["coords"]})

    label_matrix, _ = stitcher.stitch_tile_predictions(h, w, tile_results)
    n_cells = int(label_matrix.max())
    print(f"[{idx}/{len(targets)}] {stem} -> {n_cells} cellules détectées")

    colored = palette[np.clip(label_matrix, 0, 4999)]
    overlay = image.copy()
    if n_cells > 0:
        cm = label_matrix > 0
        overlay[cm] = (image[cm].astype(float) * 0.45 + colored[cm].astype(float) * 0.55).astype(np.uint8)
        for uid in range(1, n_cells + 1):
            cnts, _ = cv2.findContours((label_matrix == uid).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(overlay, cnts, -1, (255, 255, 255), 1)

    overlay_path = f"{OUTPUT_PRED_DIR}/{stem}_overlay.png"
    cv2.imwrite(overlay_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    axes[0].imshow(image)
    axes[0].set_title(f"Image Microscopique Réelle : {stem}", fontsize=13)
    axes[0].axis("off")

    axes[1].imshow(overlay)
    axes[1].set_title(f"CellOneFormer ({n_cells} cellules) : {stem}", fontsize=13, fontweight="bold")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()
